## 정수 인코딩(Integer Encoding)
- 단어를 고유한 정수로 매핑하는 방법
- 단어를 빈도수 순으로 정렬한 단어 집합(vocabulary)을 생성
- 빈도수가 높은 순서대로 낮은 숫자부터 정수를 부여

In [10]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\0627j\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\0627j\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [1]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

raw_text = "A barber is a person. a barber is good person. a barber is huge person. he Knew A Secret! The Secret He Kept is huge secret. Huge secret. His barber kept his word. a barber kept his word. His barber kept his secret. But keeping and keeping such a huge secret to himself was driving the barber crazy. the barber went up a huge mountain."

c:\Users\0627j\anaconda3\envs\NLP\lib\site-packages\sklearn\feature_extraction\image.py:167: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dtype=np.int):
c:\Users\0627j\anaconda3\envs\NLP\lib\site-packages\sklearn\linear_model\least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidanc

In [2]:
# 문장 토큰화
sentences = sent_tokenize(raw_text) # 영어 문장 규칙이 학습된 모델을 사용하여 문장별로 나눔(Mr.S 랑 I love you. 의 .은 split(".") 으로 구분 안됨.)
sentences

['A barber is a person.',
 'a barber is good person.',
 'a barber is huge person.',
 'he Knew A Secret!',
 'The Secret He Kept is huge secret.',
 'Huge secret.',
 'His barber kept his word.',
 'a barber kept his word.',
 'His barber kept his secret.',
 'But keeping and keeping such a huge secret to himself was driving the barber crazy.',
 'the barber went up a huge mountain.']

In [19]:
vocab = {}
preprocessed_sentences = []
stop_words = set(stopwords.words("english")) # 불용어(의미없거나 분석에 방해되는 단어들, a, an, the, is 등)

for sentence in sentences:
    tokenized_sentence = word_tokenize(sentence) # Don't do that. -> ["Do", "n't", "do", "that", "."] 이렇게 변경
    result = []
    for word in tokenized_sentence:
        word = word.lower()
        if word not in stop_words: # 불용어 제거
            if len(word) > 2: # 길이 2 이하의 단어 제거
                result.append(word)
                if(word not in vocab):
                    vocab[word] = 0
                vocab[word] += 1
    preprocessed_sentences.append(result)
print(preprocessed_sentences)
print(vocab)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]
{'barber': 8, 'person': 3, 'good': 1, 'huge': 5, 'knew': 1, 'secret': 6, 'kept': 4, 'word': 2, 'keeping': 2, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1}


In [33]:
vocab_sorted = sorted(vocab.items(), key=lambda x: x[1], reverse=True)
print(vocab_sorted)

word_to_index = {}
i = 0
for word, frequency in vocab_sorted:
    if frequency > 1: # 빈도수가 작은 단어 제외
        i += 1
        word_to_index[word] = i

print(word_to_index)

vocab_size = 5

# 인데스가 5 초과인 단어 제거
word_frequency = [word for word, index in word_to_index.items() if index >= vocab_size + 1]
for w in word_frequency:
    del word_to_index[w]

print(word_to_index)


# Out of Vocabulary(단어 집합에 없는 단어들)
word_to_index["OOV"] = len(word_to_index) + 1
print(word_to_index)

[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3), ('word', 2), ('keeping', 2), ('good', 1), ('knew', 1), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)]
{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7}
{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}
{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'OOV': 6}


In [38]:
encoded_sentences = []

for sentence in preprocessed_sentences:
    encoded_sentence = []
    for word in sentence:
        try: # 단어 집합에 있는 단어라면 해당 단어의 정수
            encoded_sentence.append(word_to_index[word])
        except KeyError: # 단어 집합에 없으면 OOV의 정수
            encoded_sentence.append(word_to_index["OOV"])
    encoded_sentences.append(encoded_sentence)

for i in range(len(preprocessed_sentences)):
    print(preprocessed_sentences[i], encoded_sentences[i])


['barber', 'person'] [1, 5]
['barber', 'good', 'person'] [1, 6, 5]
['barber', 'huge', 'person'] [1, 3, 5]
['knew', 'secret'] [6, 2]
['secret', 'kept', 'huge', 'secret'] [2, 4, 3, 2]
['huge', 'secret'] [3, 2]
['barber', 'kept', 'word'] [1, 4, 6]
['barber', 'kept', 'word'] [1, 4, 6]
['barber', 'kept', 'secret'] [1, 4, 2]
['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'] [6, 6, 3, 2, 6, 1, 6]
['barber', 'went', 'huge', 'mountain'] [1, 6, 3, 6]


In [48]:
# Counter 사용하기
from collections import Counter

print(preprocessed_sentences)

# import numpy as np
# words = np.hstack(preprocessed_sentences)

all_words_list = sum(preprocessed_sentences, []) # 위의 words와 동일
print(all_words_list)

vocab = Counter(all_words_list)
print(vocab)

vocab_size = 5
vocab = vocab.most_common(vocab_size) # 빈도수 높은 상위 5개 단어만 저장
print(vocab)

word_to_index = {}
i = 0
for word, frequency in vocab:
    i += 1
    word_to_index[word] = i

print(word_to_index)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]
['barber', 'person', 'barber', 'good', 'person', 'barber', 'huge', 'person', 'knew', 'secret', 'secret', 'kept', 'huge', 'secret', 'huge', 'secret', 'barber', 'kept', 'word', 'barber', 'kept', 'word', 'barber', 'kept', 'secret', 'keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy', 'barber', 'went', 'huge', 'mountain']
Counter({'barber': 8, 'secret': 6, 'huge': 5, 'kept': 4, 'person': 3, 'word': 2, 'keeping': 2, 'good': 1, 'knew': 1, 'driving': 1, 'crazy': 1, 'went': 1, 'mountain': 1})
[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3)]
{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


In [55]:
# NLTK 사용하기
from nltk import FreqDist
import numpy as np

# np.hstack으로 문장 구분을 제거(flat)
vocab = FreqDist(np.hstack(preprocessed_sentences))

print([v for v in vocab.items()])

vocab_size = 5
vocab = vocab.most_common(vocab_size)
print(vocab)

word_to_index = {word[0]: index + 1 for index, word in enumerate(vocab)}
print(word_to_index)

[('barber', 8), ('person', 3), ('good', 1), ('huge', 5), ('knew', 1), ('secret', 6), ('kept', 4), ('word', 2), ('keeping', 2), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)]
[('barber', 8), ('secret', 6), ('huge', 5), ('kept', 4), ('person', 3)]
{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5}


In [66]:
# Keras의 Tokenizer 사용
from tensorflow.keras.preprocessing.text import Tokenizer

preprocessed_sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]

tokenizer = Tokenizer()

# fit_on_texts에 코퍼스(말뭉치, 텍스트 데이터)를 입력하면 빈도수 기준으로 단어 집합 생성
tokenizer.fit_on_texts(preprocessed_sentences)

print(tokenizer.word_index)

print(tokenizer.word_counts)

print(tokenizer.texts_to_sequences(preprocessed_sentences))

print()

vocab_size = 5
tokenizer = Tokenizer(num_words = vocab_size + 1) # 제로 패딩을 추가하기 위한 전체 개수 +1
tokenizer.fit_on_texts(preprocessed_sentences)
print(tokenizer.word_index)
print(tokenizer.word_counts)

print()

vocab_size = 5
tokenizer = Tokenizer(num_words = vocab_size + 2, oov_token='OOV') # 제로패딩과 OOV를 추가하기 위해 전체 개수 + 2
tokenizer.fit_on_texts(preprocessed_sentences)
print(tokenizer.word_index)
print(tokenizer.word_counts)

print('OOV 인덱스:', tokenizer.word_index['OOV'])
print(tokenizer.texts_to_sequences(preprocessed_sentences))
# 위에서 알 수 있듯 OOV를 추가하면 tensorflow는 OOV가 1로 인코딩됨

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7, 'good': 8, 'knew': 9, 'driving': 10, 'crazy': 11, 'went': 12, 'mountain': 13}
OrderedDict([('barber', 8), ('person', 3), ('good', 1), ('huge', 5), ('knew', 1), ('secret', 6), ('kept', 4), ('word', 2), ('keeping', 2), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)])
[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]

{'barber': 1, 'secret': 2, 'huge': 3, 'kept': 4, 'person': 5, 'word': 6, 'keeping': 7, 'good': 8, 'knew': 9, 'driving': 10, 'crazy': 11, 'went': 12, 'mountain': 13}
OrderedDict([('barber', 8), ('person', 3), ('good', 1), ('huge', 5), ('knew', 1), ('secret', 6), ('kept', 4), ('word', 2), ('keeping', 2), ('driving', 1), ('crazy', 1), ('went', 1), ('mountain', 1)])

{'OOV': 1, 'barber': 2, 'secret': 3, 'huge': 4, 'kept': 5, 'person': 6, 'word': 7, 'keeping': 8, 'good': 9, 'knew': 10, 'driving': 11

## 패딩
- 병렬 연산을 위해 여러 분장의 길이를 임의로 동일하게 맞추는 작업이 필요함.

In [71]:
# Numpy 로 패딩넣기
import numpy as np 
from tensorflow.keras.preprocessing.text import Tokenizer

preprocessed_sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(preprocessed_sentences)
encoded = tokenizer.texts_to_sequences(preprocessed_sentences)
print(encoded)

max_len = max(len(item) for item in encoded)
print('최대 길이:', max_len)

for sentence in encoded:
    while len(sentence) < max_len:
        sentence.append(0)

padded_np = np.array(encoded)
print(padded_np)

[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]
최대 길이: 7
[[ 1  5  0  0  0  0  0]
 [ 1  8  5  0  0  0  0]
 [ 1  3  5  0  0  0  0]
 [ 9  2  0  0  0  0  0]
 [ 2  4  3  2  0  0  0]
 [ 3  2  0  0  0  0  0]
 [ 1  4  6  0  0  0  0]
 [ 1  4  6  0  0  0  0]
 [ 1  4  2  0  0  0  0]
 [ 7  7  3  2 10  1 11]
 [ 1 12  3 13  0  0  0]]


In [100]:
# 케라스로 패딩넣기

from tensorflow.keras.preprocessing.sequence import pad_sequences

preprocessed_sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(preprocessed_sentences)

print("tokenizer.texts_to_sequences(preprocessed_sentences)")
encoded = tokenizer.texts_to_sequences(preprocessed_sentences)
print(encoded)
print()

print("pad_sequences(encoded)")
padded = pad_sequences(encoded) # 문장의 앞에 0 추가, padding='pre' 가 기본값
print(padded)
print()

print("pad_sequences(encoded, padding='post')")
padded_0_end = pad_sequences(encoded, padding='post') # 문장 뒤에 0 추가, 
print(padded_0_end)
print()

print("pad_sequences(encoded, padding='post', maxlen=5)")
padded_max_5 = pad_sequences(encoded, padding='post', maxlen=5) # 뒤에 0 패딩
print(padded_max_5)
print()

print("pad_sequences(encoded, padding='post', truncating='pre', maxlen=5)")
padded_from_front = pad_sequences(encoded, padding='post', truncating='pre', maxlen=5) # 길이가 5보다 크면 앞쪽을 자르고 뒤에 0 패딩
print(padded_from_front)
print()

print("pad_sequences(encoded, padding='post', truncating='post', maxlen=5)")
padded_from_back = pad_sequences(encoded, padding='post', truncating='post', maxlen=5) # 길이가 5보다 크면 뒤쪽을 자르고 뒤에 0 패딩
print(padded_from_back)
print()

print("pad_sequences(encoded, padding='post', value=last_value)")
last_value = len(tokenizer.word_index) + 1 # 단어 집합의 크기보다 1 큰 숫자 사용
print(last_value)

padded_with_last_value = pad_sequences(encoded, padding='post', value=last_value) # 0 패딩이 아닌 임의의 값 사용
print(padded_with_last_value)

tokenizer.texts_to_sequences(preprocessed_sentences)
[[1, 5], [1, 8, 5], [1, 3, 5], [9, 2], [2, 4, 3, 2], [3, 2], [1, 4, 6], [1, 4, 6], [1, 4, 2], [7, 7, 3, 2, 10, 1, 11], [1, 12, 3, 13]]

pad_sequences(encoded)
[[ 0  0  0  0  0  1  5]
 [ 0  0  0  0  1  8  5]
 [ 0  0  0  0  1  3  5]
 [ 0  0  0  0  0  9  2]
 [ 0  0  0  2  4  3  2]
 [ 0  0  0  0  0  3  2]
 [ 0  0  0  0  1  4  6]
 [ 0  0  0  0  1  4  6]
 [ 0  0  0  0  1  4  2]
 [ 7  7  3  2 10  1 11]
 [ 0  0  0  1 12  3 13]]

pad_sequences(encoded, padding='post')
[[ 1  5  0  0  0  0  0]
 [ 1  8  5  0  0  0  0]
 [ 1  3  5  0  0  0  0]
 [ 9  2  0  0  0  0  0]
 [ 2  4  3  2  0  0  0]
 [ 3  2  0  0  0  0  0]
 [ 1  4  6  0  0  0  0]
 [ 1  4  6  0  0  0  0]
 [ 1  4  2  0  0  0  0]
 [ 7  7  3  2 10  1 11]
 [ 1 12  3 13  0  0  0]]

pad_sequences(encoded, padding='post', maxlen=5)
[[ 1  5  0  0  0]
 [ 1  8  5  0  0]
 [ 1  3  5  0  0]
 [ 9  2  0  0  0]
 [ 2  4  3  2  0]
 [ 3  2  0  0  0]
 [ 1  4  6  0  0]
 [ 1  4  6  0  0]
 [ 1  4  2  0  0]
 [ 3  